In [ ]:
from google.cloud import pubsub_v1
from google.cloud import bigquery
import datetime

In [ ]:
client = bigquery.Client()

dataset_id = f"{client.project}.flight_streaming"
dataset = bigquery.Dataset(dataset_id)
dataset.location = "US"

dataset = client.create_dataset(dataset, exists_ok=True)
print("Dataset created:", dataset_id)

Dataset created: qwiklabs-gcp-02-cb7e0bebe617.flight_streaming


In [ ]:
schema = [
    bigquery.SchemaField("MT", "STRING"),
    bigquery.SchemaField("TT", "INT64"),
    bigquery.SchemaField("SID", "STRING"),
    bigquery.SchemaField("AID", "STRING"),
    bigquery.SchemaField("Hex", "STRING"),
    bigquery.SchemaField("FID", "STRING"),
    bigquery.SchemaField("DMG", "DATE"),
    bigquery.SchemaField("TMG", "TIME"),
    bigquery.SchemaField("DML", "DATE"),
    bigquery.SchemaField("TML", "TIME"),
    bigquery.SchemaField("CS", "STRING"),
    bigquery.SchemaField("Alt", "INT64"),
    bigquery.SchemaField("GS", "INT64"),
    bigquery.SchemaField("Trk", "INT64"),
    bigquery.SchemaField("Lat", "FLOAT64"),
    bigquery.SchemaField("Lng", "FLOAT64"),
    bigquery.SchemaField("VR", "INT64"),
    bigquery.SchemaField("Sq", "STRING"),
    bigquery.SchemaField("Alrt", "INT64"),
    bigquery.SchemaField("Emer", "INT64"),
    bigquery.SchemaField("SPI", "INT64"),
    bigquery.SchemaField("Gnd", "INT64"),
]

table_id = f"{client.project}.flight_streaming.flight_transponder_data"
table = bigquery.Table(table_id, schema=schema)
table = client.create_table(table, exists_ok=True)
print("Table created:", table_id)

Table created: qwiklabs-gcp-02-cb7e0bebe617.flight_streaming.flight_transponder_data


In [7]:
subscriber = pubsub_v1.SubscriberClient()
subscription_path = subscriber.subscription_path(
    client.project, "flight-transponder-sub"
)

topic_path = "projects/paul-leroy/topics/flight-transponder"

subscriber.create_subscription(
    name=subscription_path,
    topic=topic_path
)

name: "projects/qwiklabs-gcp-02-cb7e0bebe617/subscriptions/flight-transponder-sub"
topic: "projects/paul-leroy/topics/flight-transponder"
push_config {
}
ack_deadline_seconds: 10
message_retention_duration {
  seconds: 604800
}
expiration_policy {
  ttl {
    seconds: 2678400
  }
}
state: ACTIVE

In [9]:
def callback(message):
    row = message.data.decode("utf-8").split(",")

    if len(row) != 22:
        message.ack()
        return

    record = {
        "MT": row[0],
        "TT": int(row[1]) if row[1] else None,
        "SID": row[2],
        "AID": row[3],
        "Hex": row[4],
        "FID": row[5],
        "DMG": row[6],
        "TMG": row[7].split(".")[0] if row[7] else None,
        "DML": row[8],
        "TML": row[9].split(".")[0] if row[9] else None,
        "CS": row[10],
        "Alt": int(row[11]) if row[11] else None,
        "GS": int(row[12]) if row[12] else None,
        "Trk": int(row[13]) if row[13] else None,
        "Lat": float(row[14]) if row[14] else None,
        "Lng": float(row[15]) if row[15] else None,
        "VR": int(row[16]) if row[16] else None,
        "Sq": row[17],
        "Alrt": int(row[18]) if row[18] else None,
        "Emer": int(row[19]) if row[19] else None,
        "SPI": int(row[20]) if row[20] else None,
        "Gnd": int(row[21]) if row[21] else None,
    }

    errors = client.insert_rows_json(table_id, [record])
    if not errors:
        message.ack()

In [10]:
streaming_pull_future = subscriber.subscribe(subscription_path, callback=callback)
print("Listening for messages...")

Listening for messages...


In [11]:
%%bigquery
SELECT COUNT(*) AS total_records
FROM `flight_streaming.flight_transponder_data`;

Query is running:   0%|          |

Downloading:   0%|          |

,total_records
0,0


In [12]:
def callback(message):
    print("RAW MESSAGE:", message.data.decode("utf-8"))
    message.ack()

In [13]:
streaming_pull_future = subscriber.subscribe(
    subscription_path,
    callback=callback
)

print("Listening for messages...")

Listening for messages...


In [15]:
subscriber.create_subscription(
    name=subscription_path,
    topic="projects/paul-leroy/topics/flight-transponder"
)

RAW MESSAGE: MSG,5,1,1,48C22C,1,2026/03/27,08:03:31.768,2026/03/27,08:03:31.800,,18400,,,,,,,0,,0,
RAW MESSAGE: MSG,5,1,1,407995,1,2026/03/27,08:03:31.768,2026/03/27,08:03:31.800,,7025,,,,,,,0,,0,
RAW MESSAGE: MSG,3,1,1,400B84,1,2026/03/27,08:03:31.769,2026/03/27,08:03:31.800,,12975,,,51.76879,-0.13847,,,0,,0,0
RAW MESSAGE: MSG,5,1,1,8964B4,1,2026/03/27,08:03:31.770,2026/03/27,08:03:31.800,,9000,,,,,,,0,,0,
RAW MESSAGE: MSG,3,1,1,407995,1,2026/03/27,08:03:31.772,2026/03/27,08:03:31.800,,7025,,,51.36099,-0.27919,,,0,,0,0
RAW MESSAGE: MSG,5,1,1,400B84,1,2026/03/27,08:03:31.772,2026/03/27,08:03:31.801,,12975,,,,,2272,,0,,0,
RAW MESSAGE: MSG,8,1,1,43C208,1,2026/03/27,08:03:31.773,2026/03/27,08:03:31.801,,,,,,,,,,,,0
RAW MESSAGE: MSG,7,1,1,4075FF,1,2026/03/27,08:03:31.773,2026/03/27,08:03:31.801,,13100,,,,,,,,,,
RAW MESSAGE: MSG,5,1,1,4CA802,1,2026/03/27,08:03:31.773,2026/03/27,08:03:31.801,,38000,400,317,,,,,0,,0,
RAW MESSAGE: MSG,6,1,1,4CACB6,1,2026/03/27,08:03:31.773,2026/03/27,08:03:31.

AlreadyExists: 409 Resource already exists in the project (resource=flight-transponder-sub).

RAW MESSAGE: MSG,8,1,1,4075FF,1,2026/03/27,08:04:23.631,2026/03/27,08:04:23.681,,,,,,,,,,,,0
RAW MESSAGE: MSG,5,1,1,407569,1,2026/03/27,08:04:23.632,2026/03/27,08:04:23.681,,23375,,,,,,,0,,0,
RAW MESSAGE: MSG,8,1,1,405B66,1,2026/03/27,08:04:23.633,2026/03/27,08:04:23.681,,,,,,,,,,,,0
RAW MESSAGE: MSG,8,1,1,408229,1,2026/03/27,08:04:23.634,2026/03/27,08:04:23.681,,,,,,,,,,,,0
RAW MESSAGE: MSG,8,1,1,4079F7,1,2026/03/27,08:04:23.634,2026/03/27,08:04:23.681,,,,,,,,,,,,0
RAW MESSAGE: MSG,5,1,1,8964B4,1,2026/03/27,08:04:23.634,2026/03/27,08:04:23.682,,9000,256,59,,,,,0,,0,
RAW MESSAGE: MSG,7,1,1,408032,1,2026/03/27,08:04:23.635,2026/03/27,08:04:23.682,,9100,,,,,,,,,,
RAW MESSAGE: MSG,7,1,1,484EE5,1,2026/03/27,08:04:23.636,2026/03/27,08:04:23.682,,24000,,,,,,,,,,
RAW MESSAGE: MSG,8,1,1,408227,1,2026/03/27,08:04:23.637,2026/03/27,08:04:23.682,,,,,,,,,,,,0
RAW MESSAGE: MSG,7,1,1,400A0E,1,2026/03/27,08:04:23.637,2026/03/27,08:04:23.682,,14000,,,,,,,,,,
RAW MESSAGE: MSG,7,1,1,405A47,1,2026/03/27,

In [16]:
def callback(message):
    data = message.data.decode("utf-8")
    fields = data.split(",")

    print("FIELDS COUNT:", len(fields))

    if len(fields) != 22:
        print("SKIPPED:", data)
        message.ack()
        return

    message.ack()

RAW MESSAGE:RAW MESSAGE: MSG,7,1,1,A54137,1,2026/03/27,08:04:54.511,2026/03/27,08:04:54.540,,8000,,,,,,,,,,
RAW MESSAGE: MSG,6,1,1,4007F6,1,2026/03/27,08:04:54.511,2026/03/27,08:04:54.540,,,,,,,-800,5163,0,0,0,
RAW MESSAGE: MSG,6,1,1,4007F6,1,2026/03/27,08:04:54.512,2026/03/27,08:04:54.540,,,284,143,,,,5163,0,0,0,
RAW MESSAGE: MSG,7,1,1,4008B3,1,2026/03/27,08:04:54.512,2026/03/27,08:04:54.540,,15850,,,,,,,,,,
RAW MESSAGE: MSG,7,1,1,710119,1,2026/03/27,08:04:54.513,2026/03/27,08:04:54.540,,7250,,,,,,,,,,
RAW MESSAGE: MSG,8,1,1,407898,1,2026/03/27,08:04:54.513,2026/03/27,08:04:54.540,,,,,,,,,,,,0
RAW MESSAGE: MSG,7,1,1,484EE5,1,2026/03/27,08:04:54.513,2026/03/27,08:04:54.540,,24000,,,,,,,,,,
RAW MESSAGE: MSG,5,1,1,406222,1,2026/03/27,08:04:54.514,2026/03/27,08:04:54.540,,8975,,,,,-384,,0,,0,
RAW MESSAGE: MSG,7,1,1,4CADF7,1,2026/03/27,08:04:54.514,2026/03/27,08:04:54.540,,26950,,,,,,,,,,
RAW MESSAGE: MSG,7,1,1,781B2B,1,2026/03/27,08:04:54.515,2026/03/27,08:04:54.540,,29975,,,,,,,,,,
RAW M

In [17]:
errors = client.insert_rows_json(table_id, [record])

if errors:
    print("BQ INSERT ERROR:", errors)
else:
    print("INSERTED ROW")
    message.ack()

RAW MESSAGE: MSG,4,1,1,405B64,1,2026/03/27,08:06:40.734,2026/03/27,08:06:40.755,,,462,145,,,0,,,,,0
RAW MESSAGE: MSG,5,1,1,4064BB,1,2026/03/27,08:06:42.173,2026/03/27,08:06:42.183,,29100,,,,,,,0,,0,
RAW MESSAGE: MSG,7,1,1,4D2453,1,2026/03/27,08:06:42.174,2026/03/27,08:06:42.183,,13850,,,,,,,,,,
RAW MESSAGE: MSG,8,1,1,400E09,1,2026/03/27,08:06:42.174,2026/03/27,08:06:42.183,,,,,,,,,,,,0
RAW MESSAGE: MSG,6,1,1,4075FD,1,2026/03/27,08:06:42.175,2026/03/27,08:06:42.183,,,,,,,,4225,0,0,0,
RAW MESSAGE: MSG,5,1,1,710119,1,2026/03/27,08:06:42.175,2026/03/27,08:06:42.183,,6200,,,,,,,0,,0,
RAW MESSAGE: MSG,6,1,1,4CA8FB,1,2026/03/27,08:06:42.175,2026/03/27,08:06:42.183,,,,,,,,2216,0,0,0,
RAW MESSAGE: MSG,6,1,1,4CA8FB,1,2026/03/27,08:06:42.175,2026/03/27,08:06:42.183,NOS461  ,,,,,,,2216,0,0,0,
RAW MESSAGE: MSG,5,1,1,4075FF,1,2026/03/27,08:06:42.177,2026/03/27,08:06:42.183,,21250,,,,,,,0,,0,
RAW MESSAGE: MSG,8,1,1,AC5C1F,1,2026/03/27,08:06:42.177,2026/03/27,08:06:42.183,,,,,,,,,,,,0
RAW MESSAGE: MSG

NameError: name 'record' is not defined

RAW MESSAGE: MSG,4,1,1,405B66,1,2026/03/27,08:06:40.757,2026/03/27,08:06:40.782,,,365,133,,,64,,,,,
RAW MESSAGE: MSG,5,1,1,484EE5,1,2026/03/27,08:06:40.758,2026/03/27,08:06:40.782,,21150,370,328,,,,,0,,0,


In [18]:
def callback(message):
    row = message.data.decode("utf-8").split(",")

    if len(row) != 22:
        message.ack()
        return

    record = {
        "MT": row[0],
        "TT": int(row[1]) if row[1] else None,
        "SID": row[2],
        "AID": row[3],
        "Hex": row[4],
        "FID": row[5],
        "DMG": row[6].replace("/", "-") if row[6] else None,
        "TMG": row[7].split(".")[0] if row[7] else None,
        "DML": row[8].replace("/", "-") if row[8] else None,
        "TML": row[9].split(".")[0] if row[9] else None,
        "CS": row[10],
        "Alt": int(row[11]) if row[11] else None,
        "GS": int(row[12]) if row[12] else None,
        "Trk": int(row[13]) if row[13] else None,
        "Lat": float(row[14]) if row[14] else None,
        "Lng": float(row[15]) if row[15] else None,
        "VR": int(row[16]) if row[16] else None,
        "Sq": row[17],
        "Alrt": int(row[18]) if row[18] else None,
        "Emer": int(row[19]) if row[19] else None,
        "SPI": int(row[20]) if row[20] else None,
        "Gnd": int(row[21]) if row[21] else None,
    }

    errors = client.insert_rows_json(table_id, [record])

    if not errors:
        message.ack()

RAW MESSAGE: MSG,3,1,1,3C61F3,1,2026/03/27,08:06:52.068,2026/03/27,08:06:52.090,,31000,,,51.877069,-3.392868,,,0,,0,0
RAW MESSAGE: MSG,5,1,1,4CAD13,1,2026/03/27,08:08:15.670,2026/03/27,08:08:15.682,,4775,,,,,,,0,,0,
RAW MESSAGE: MSG,4,1,1,4075FD,1,2026/03/27,08:08:15.672,2026/03/27,08:08:15.682,,,346,352,,,3136,,,,,0
RAW MESSAGE: MSG,8,1,1,A8E4E9,1,2026/03/27,08:08:15.673,2026/03/27,08:08:15.682,,,,,,,,,,,,0
RAW MESSAGE: MSG,7,1,1,4D02CD,1,2026/03/27,08:08:15.674,2026/03/27,08:08:15.682,,3475,,,,,,,,,,
RAW MESSAGE: MSG,8,1,1,400E09,1,2026/03/27,08:08:15.675,2026/03/27,08:08:15.682,,,,,,,,,,,,0
RAW MESSAGE: MSG,4,1,1,407C88,1,2026/03/27,08:08:15.675,2026/03/27,08:08:15.682,,,219,115,,,-448,,,,,0
RAW MESSAGE: MSG,6,1,1,4079CC,1,2026/03/27,08:08:15.676,2026/03/27,08:08:15.682,,,,,,,,3256,0,0,0,
RAW MESSAGE: MSG,8,1,1,4008B3,1,2026/03/27,08:08:15.677,2026/03/27,08:08:15.682,,,,,,,,,,,,0
RAW MESSAGE: MSG,8,1,1,781B2B,1,2026/03/27,08:08:15.677,2026/03/27,08:08:15.682,,,,,,,,,,,,0
RAW MESSAGE

In [19]:
SELECT COUNT(*) AS total_records
FROM `flight_streaming.flight_transponder_data`;

RAW MESSAGE: MSG,5,1,1,39C425,1,2026/03/27,08:09:23.509,2026/03/27,08:09:23.563,,36975,,,,,,,0,,0,
RAW MESSAGE: MSG,8,1,1,4CA802,1,2026/03/27,08:09:23.510,2026/03/27,08:09:23.564,,,,,,,,,,,,
RAW MESSAGE: MSG,8,1,1,AA5F41,1,2026/03/27,08:09:23.510,2026/03/27,08:09:23.564,,,,,,,,,,,,0
RAW MESSAGE: MSG,8,1,1,A8C88F,1,2026/03/27,08:09:23.510,2026/03/27,08:09:23.564,,,,,,,,,,,,0
RAW MESSAGE: MSG,8,1,1,4CA8FB,1,2026/03/27,08:09:23.511,2026/03/27,08:09:23.564,,,,,,,,,,,,0
RAW MESSAGE: MSG,5,1,1,407898,1,2026/03/27,08:09:23.512,2026/03/27,08:09:23.564,,8000,,,,,,,0,,0,
RAW MESSAGE: MSG,6,1,1,407898,1,2026/03/27,08:09:23.513,2026/03/27,08:09:23.564,,,,,,,,5171,0,0,0,
RAW MESSAGE: MSG,5,1,1,406250,1,2026/03/27,08:09:23.513,2026/03/27,08:09:23.564,,5800,,,,,,,0,,0,
RAW MESSAGE: MSG,7,1,1,405B64,1,2026/03/27,08:09:23.514,2026/03/27,08:09:23.564,,37000,,,,,,,,,,
RAW MESSAGE: MSG,5,1,1,4CACB6,1,2026/03/27,08:09:23.515,2026/03/27,08:09:23.564,,12000,,,,,,,0,,0,
RAW MESSAGE: MSG,5,1,1,400F99,1,2026/03

SyntaxError: Invalid star expression (2204215912.py, line 1)

RAW MESSAGE: MSG,8,1,1,A4CE71,1,2026/03/27,08:09:17.692,2026/03/27,08:09:17.716,,,,,,,,,,,,0


In [20]:
SELECT *
FROM `flight_streaming.flight_transponder_data`
LIMIT 5;

RAW MESSAGE:RAW MESSAGE: MSG,7,1,1,400F99,1,2026/03/27,08:11:49.602,2026/03/27,08:11:49.655,,5800,,,,,,,,,,
RAW MESSAGE: MSG,5,1,1,40815B,1,2026/03/27,08:11:49.602,2026/03/27,08:11:49.655,,27000,,,,,,,0,,0,
RAW MESSAGE: MSG,7,1,1,405D0F,1,2026/03/27,08:11:49.604,2026/03/27,08:11:49.655,,7950,,,,,,,,,,
RAW MESSAGE: MSG,6,1,1,4D227F,1,2026/03/27,08:11:49.605,2026/03/27,08:11:49.655,,,,,,,,4032,0,0,0,
RAW MESSAGE: MSG,6,1,1,AA5F41,1,2026/03/27,08:11:49.605,2026/03/27,08:11:49.655,,,250,42,,,,5172,0,0,0,
RAW MESSAGE: MSG,5,1,1,406250,1,2026/03/27,08:11:49.607,2026/03/27,08:11:49.655,,10175,,,,,,,0,,0,
RAW MESSAGE: MSG,7,1,1,4007F6,1,2026/03/27,08:11:49.609,2026/03/27,08:11:49.655,,6650,,,,,,,,,,
RAW MESSAGE: MSG,8,1,1,440CF8,1,2026/03/27,08:11:49.610,2026/03/27,08:11:49.655,,,,,,,,,,,,0
RAW MESSAGE: MSG,7,1,1,407C88,1,2026/03/27,08:11:49.610,2026/03/27,08:11:49.655,,8875,,,,,,,,,,
RAW MESSAGE: MSG,5,1,1,4BAB23,1,2026/03/27,08:11:49.611,2026/03/27,08:11:49.655,,35000,,,,,,,0,,0,
RAW MESSAGE

SyntaxError: invalid syntax (1172271861.py, line 1)

RAW MESSAGE: MSG,5,1,1,407837,1,2026/03/27,08:11:49.651,2026/03/27,08:11:49.658,,2600,,,,,,,0,,0,
RAW MESSAGE: MSG,7,1,1,C058BE,1,2026/03/27,08:11:49.651,2026/03/27,08:11:49.658,,9000,,,,,,,,,,
RAW MESSAGE: MSG,5,1,1,4080EA,1,2026/03/27,08:11:49.651,2026/03/27,08:11:49.658,,34700,,,,,-1120,,0,,0,
RAW MESSAGE: MSG,7,1,1,394A15,1,2026/03/27,08:11:49.652,2026/03/27,08:11:49.658,,32000,,,,,,,,,,
RAW MESSAGE: MSG,8,1,1,405B64,1,2026/03/27,08:11:49.652,2026/03/27,08:11:49.658,,,,,,,,,,,,0
RAW MESSAGE: MSG,5,1,1,406279,1,2026/03/27,08:11:49.653,2026/03/27,08:11:49.658,,8650,,,,,,,0,,0,
RAW MESSAGE: MSG,4,1,1,405B64,1,2026/03/27,08:11:49.654,2026/03/27,08:11:49.658,,,458,141,,,-64,,,,,0
RAW MESSAGE: MSG,5,1,1,8964B4,1,2026/03/27,08:11:49.654,2026/03/27,08:11:49.709,,5325,,,,,,,0,,0,
RAW MESSAGE: MSG,7,1,1,4D2453,1,2026/03/27,08:11:49.655,2026/03/27,08:11:49.709,,5900,,,,,,,,,,
RAW MESSAGE: MSG,5,1,1,AB9D9E,1,2026/03/27,08:11:49.655,2026/03/27,08:11:49.709,,2800,164,270,,,,,0,,0,
RAW MESSAGE: M

In [21]:
SELECT
  ST_GEOGPOINT(Lng, Lat) AS location
FROM `flight_streaming.flight_transponder_data`
WHERE Lat IS NOT NULL
  AND Lng IS NOT NULL;

RAW MESSAGE: MSG,8,1,1,4CAA55,1,2026/03/27,08:12:01.531,2026/03/27,08:12:01.556,,,,,,,,,,,,0
RAW MESSAGE: MSG,7,1,1,4077E0,1,2026/03/27,08:12:55.514,2026/03/27,08:12:55.522,,8575,,,,,,,,,,
RAW MESSAGE: MSG,8,1,1,408227,1,2026/03/27,08:12:55.516,2026/03/27,08:12:55.522,,,,,,,,,,,,0
RAW MESSAGE: MSG,5,1,1,406A3D,1,2026/03/27,08:12:55.516,2026/03/27,08:12:55.522,BAW754B ,10975,,,,,,,0,,0,
RAW MESSAGE: MSG,3,1,1,394A15,1,2026/03/27,08:12:55.516,2026/03/27,08:12:55.522,,32000,,,50.76861,1.19102,,,0,,0,0
RAW MESSAGE: MSG,8,1,1,40815B,1,2026/03/27,08:12:55.517,2026/03/27,08:12:55.522,,,,,,,,,,,,0
RAW MESSAGE: MSG,8,1,1,484369,1,2026/03/27,08:12:55.517,2026/03/27,08:12:55.522,,,,,,,,,,,,0
RAW MESSAGE: MSG,5,1,1,408227,1,2026/03/27,08:12:55.517,2026/03/27,08:12:55.522,,7775,,,,,-512,,0,,0,
RAW MESSAGE: MSG,8,1,1,407537,1,2026/03/27,08:12:55.517,2026/03/27,08:12:55.522,,,,,,,,,,,,0
RAW MESSAGE: MSG,8,1,1,461F50,1,2026/03/27,08:12:55.518,2026/03/27,08:12:55.573,,,,,,,,,,,,0
RAW MESSAGE: MSG,8,1,1

IndentationError: unexpected indent (60758051.py, line 2)

RAW MESSAGE: MSG,8,1,1,40756F,1,2026/03/27,08:12:01.545,2026/03/27,08:12:01.556,,,,,,,,,,,,0
RAW MESSAGE: MSG,8,1,1,4ACA45,1,2026/03/27,08:12:55.679,2026/03/27,08:12:55.686,,,,,,,,,,,,0
RAW MESSAGE: MSG,4,1,1,4010EB,1,2026/03/27,08:12:55.680,2026/03/27,08:12:55.686,,,473,163,,,0,,,,,0
RAW MESSAGE: MSG,8,1,1,4080E0,1,2026/03/27,08:12:55.680,2026/03/27,08:12:55.686,,,,,,,,,,,,0
RAW MESSAGE: MSG,8,1,1,43C208,1,2026/03/27,08:12:55.680,2026/03/27,08:12:55.686,,,,,,,,,,,,0
RAW MESSAGE: MSG,4,1,1,AB9383,1,2026/03/27,08:12:55.681,2026/03/27,08:12:55.686,,,381,148,,,-1152,,,,,0
RAW MESSAGE: MSG,7,1,1,3C49C8,1,2026/03/27,08:12:55.683,2026/03/27,08:12:55.737,,9000,,,,,,,,,,
RAW MESSAGE: MSG,8,1,1,43C208,1,2026/03/27,08:12:55.683,2026/03/27,08:12:55.737,,,,,,,,,,,,0
RAW MESSAGE: MSG,8,1,1,406016,1,2026/03/27,08:12:55.683,2026/03/27,08:12:55.737,,,,,,,,,,,,0
RAW MESSAGE: MSG,8,1,1,4079F7,1,2026/03/27,08:12:55.684,2026/03/27,08:12:55.737,,,,,,,,,,,,0
RAW MESSAGE: MSG,8,1,1,4D24C2,1,2026/03/27,08:12: